# [SOLUTION] UdaPlay Project

## Part 02 - Agent

This notebook turns the vector database from Part 01 into **UdaPlay**, a stateful
research agent for the video game industry. The agent:

1. Answers questions from internal knowledge (RAG over the `udaplay` collection)
2. Judges whether that internal knowledge is actually good enough
3. Falls back to the web (Tavily) when it is not
4. Keeps conversation state across turns, and long-term memory across sessions
5. Returns a natural-language answer **and** a validated JSON report with citations

Part 01 must be run first - it creates the persistent ChromaDB collection this
notebook opens.

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import json
import os
import time
from typing import Any, Dict, List, Optional, TypedDict, Union

from pydantic import BaseModel
from tavily import TavilyClient

from lib.messages import AIMessage, SystemMessage, ToolMessage, UserMessage
from lib.memory import ShortTermMemory
from lib.parsers import JsonOutputParser, PydanticOutputParser
from lib.state_machine import EntryPoint, Run, StateMachine, Step, Termination
from lib.tooling import Tool, ToolCall, tool
from lib.udaplay import (
    DEFAULT_LLM_MODEL,
    EvaluationReport,
    GameReport,
    GameVectorStore,
    LongTermMemoryStore,
    SafeLLM,
    get_embedding_function,
    load_config,
    truncate,
)

In [3]:
config = load_config()

Credentials loaded from config.env
  OPENAI_API_KEY  : voc-21...9965 (49 chars)
  TAVILY_API_KEY  : tvly-d...siQE (58 chars)
  OPENAI_BASE_URL : https://openai.vocareum.com/v1


### Knowledge sources

Two persistent collections, both opened with the **same** embedding function:

| Collection | Contents | Written by |
|---|---|---|
| `udaplay` | the 25 curated game records | Part 01 |
| `udaplay_long_term_memory` | facts learned from web searches | this agent, at run time |

The second one is what makes the agent improve over time: anything it has to look up on
the web is written back, so a later session can answer the same question from memory.

In [4]:
embedding_fn = get_embedding_function()

game_store = GameVectorStore(embedding_function=embedding_fn, reset=False)
long_term_memory = LongTermMemoryStore(embedding_function=embedding_fn)

if game_store.count() == 0:
    raise RuntimeError("The `udaplay` collection is empty - run Udaplay_01 first.")

Embedding function: OpenAI `text-embedding-3-small` (1536 dimensions)
Collection `udaplay` ready with 25 documents
Long-term memory ready with 0 fragments


### Tools

Four tools. The three required ones, plus a memory lookup so the agent can reuse what it
already learned instead of paying for the same web search twice.

| Tool | Purpose |
|---|---|
| `retrieve_game` | semantic search over the internal game collection |
| `evaluate_retrieval` | LLM-as-judge verdict on whether those documents answer the question |
| `game_web_search` | Tavily web search, with the findings written to long-term memory |
| `recall_learned_facts` | semantic search over previously learned web findings |

Each docstring is written as a *routing prompt*: it is the only thing the model sees when
deciding which tool to call and what to pass it, so it states what the tool does, what
the arguments mean, and what comes back.

#### Retrieve Game Tool

In [5]:
RETRIEVE_N_RESULTS = 5


@tool
def retrieve_game(query: str) -> List[Dict[str, Any]]:
    """Semantic search: finds the most relevant games in the internal vector DB.

    This is the primary knowledge source and should be tried first for any question
    about a specific game, platform, publisher or release year.

    args:
    - query: a question about the game industry, in natural language.

    You'll receive results as a list. Each element contains:
    - Name: name of the game
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - YearOfRelease: year when that game was released for that platform
    - Genre: genre of the game
    - Publisher: company that published the game
    - Description: additional details about the game
    - source_id: id of the document, cite it in your answer like [006]
    - similarity: 0.0-1.0 match score; anything below ~0.3 is probably unrelated
    """
    hits = game_store.search(query, n_results=RETRIEVE_N_RESULTS)
    return [hit.model_dump() for hit in hits]

#### Evaluate Retrieval Tool

Vector search always returns its nearest neighbours, even when nothing relevant exists -
Part 01 showed it confidently returning three games for a Mortal Kombat X question. So a
second model reads the question and the documents and decides whether they actually
contain the answer.

The verdict is parsed into the `EvaluationReport` schema (`useful`, `confidence`,
`description`, `missing_information`) rather than free text, so the agent gets a boolean
it can branch on. If structured output is unavailable on the endpoint, the tool retries
with a plain JSON prompt, and if that fails too it returns `useful=False` - failing
towards a web search is the safe direction.

In [6]:
JUDGE_SYSTEM_PROMPT = (
    "You are a strict retrieval evaluator for a video game research assistant. "
    "Your task is to evaluate if the documents are enough to respond to the query. "
    "Give a detailed explanation, so it's possible to take an action to accept it or not. "
    "Only mark the documents as useful when they directly contain the facts needed to "
    "answer the question. Near-misses do not count: a document about a different game, a "
    "different platform, or a different entry in the same series is NOT enough. "
    "If the question asks about current, ongoing, upcoming or unreleased things, the "
    "documents are never enough, because they are a static snapshot. "
    "If the question does not identify which game it is about, the documents are not "
    "enough, no matter how many games they describe. "
    "Do not demand more precision than the question asks for: if the user asks when a "
    "game was released and the document gives the release year, that is enough. Only "
    "require an exact day or month when the user explicitly asked for one."
)


@tool
def evaluate_retrieval(question: str, retrieved_docs: List[str]) -> Dict[str, Any]:
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.

    Always call this after `retrieve_game`, before answering.

    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector
      Database. Pass each document as a text line containing the game's name, platform,
      year, publisher and description.

    The result includes:
    - useful: whether the documents are useful to answer the question
    - confidence: 0.0-1.0 confidence in that verdict
    - description: description about the evaluation result
    - missing_information: what is missing, when the documents are not enough
    """
    if not retrieved_docs:
        return EvaluationReport(
            useful=False,
            confidence=1.0,
            description="No documents were retrieved, so the internal database cannot answer this.",
            missing_information="Everything - the internal search returned nothing.",
        ).model_dump()

    documents = "\n".join(f"- {doc}" for doc in retrieved_docs)
    prompt = (
        f"{JUDGE_SYSTEM_PROMPT}\n\n"
        f"# Question\n{question}\n\n"
        f"# Retrieved documents\n{documents}\n\n"
        f"# Your verdict"
    )

    judge = SafeLLM(model=DEFAULT_LLM_MODEL, temperature=0.0)

    # Preferred path: structured outputs, validated against the pydantic schema.
    try:
        response = judge.invoke(prompt, response_format=EvaluationReport)
        return PydanticOutputParser(model_class=EvaluationReport).parse(response).model_dump()
    except Exception:
        pass  # fall through to the JSON fallback below

    # Fallback: ask for raw JSON if the endpoint does not support structured outputs.
    try:
        response = judge.invoke(
            prompt + "\n\nReply with ONLY a JSON object with keys: "
            "useful (boolean), confidence (number 0-1), description (string), "
            "missing_information (string). No markdown, no backticks."
        )
        payload = JsonOutputParser().parse(response)
        return EvaluationReport(**payload).model_dump()
    except Exception as json_error:
        # Fail towards a web search rather than towards a confident wrong answer.
        return EvaluationReport(
            useful=False,
            confidence=0.0,
            description=f"The evaluator could not be reached ({type(json_error).__name__}). "
                        f"Treating the retrieval as insufficient.",
            missing_information="Unverified retrieval.",
        ).model_dump()

#### Game Web Search Tool

The fallback. Tavily returns a synthesised answer plus the pages it came from, and the
tool keeps the URLs so the agent can cite them.

It also *learns*: every finding is written to the long-term memory collection, keyed by a
hash of its content so repeated searches do not pile up duplicates.

In [7]:
WEB_MAX_RESULTS = 4
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


@tool
def game_web_search(question: str) -> Dict[str, Any]:
    """Search the web for video game information that the internal database lacks.

    Use this only after `retrieve_game` and `evaluate_retrieval` have shown the internal
    documents are not enough, or when the question is about current, ongoing, upcoming or
    very recent events, which the static internal database cannot cover.

    args:
    - question: a question about the game industry, in natural language.

    Returns:
    - answer: a short synthesised answer from the web
    - results: list of sources, each with title, url and an extract. Cite the url.
    """
    try:
        raw = tavily_client.search(
            query=question,
            max_results=WEB_MAX_RESULTS,
            include_answer=True,
            search_depth="basic",
        )
    except Exception as exc:
        return {"error": f"Web search failed ({type(exc).__name__}: {exc})"}

    results = [
        {
            "title": item.get("title", ""),
            "url": item.get("url", ""),
            "content": truncate(item.get("content", ""), 500),
        }
        for item in raw.get("results", [])
    ]
    payload = {
        "question": question,
        "answer": raw.get("answer", "") or "",
        "results": results,
    }

    # Learn from the search: persist the finding for future sessions.
    if payload["answer"]:
        fragment_id = long_term_memory.remember(
            content=f"{question} -> {payload['answer']}",
            source=results[0]["url"] if results else "tavily",
            question=question,
        )
        payload["stored_as"] = fragment_id

    return payload

#### Recall Learned Facts Tool

The read side of long-term memory. Checking here before hitting the web is cheaper and
faster, and it is what makes the "agent learns from web searches" behaviour visible: ask
the same question in a new session and it comes back from memory with a `mem-...`
citation.

In [8]:
@tool
def recall_learned_facts(query: str) -> List[Dict[str, Any]]:
    """Search the agent's long-term memory of facts learned during previous web searches.

    Check this before running a new web search - the answer may already be known.
    Memory is not authoritative: if a fragment looks stale or only partly relevant,
    still run `game_web_search`.

    args:
    - query: a question about the game industry, in natural language.

    Returns a list of fragments, each with:
    - id: memory id, cite it like [mem-1a2b3c]
    - content: the remembered fact
    - source: where it originally came from
    - similarity: 0.0-1.0 match score
    """
    return long_term_memory.recall(query, n_results=3)

### Tool smoke tests

In [9]:
docs = retrieve_game("When was Pokémon Gold and Silver released?")
for doc in docs[:3]:
    print(f"[{doc['source_id']}] sim={doc['similarity']:.3f}  {doc['Name']} "
          f"({doc['YearOfRelease']}, {doc['Platform']})")

print("\nEvaluation of a question the database CAN answer:")
verdict = evaluate_retrieval(
    question="When was Pokémon Gold and Silver released?",
    retrieved_docs=[f"{d['Name']} ({d['YearOfRelease']}, {d['Platform']}) - {d['Description']}" for d in docs],
)
print(json.dumps(verdict, indent=2))

[006] sim=0.692  Pokémon Gold and Silver (1999, Game Boy Color)
[017] sim=0.556  Pokémon Red and Blue (1996, Game Boy)
[007] sim=0.543  Pokémon Ruby and Sapphire (2002, Game Boy Advance)

Evaluation of a question the database CAN answer:
{
  "useful": true,
  "confidence": 0.9,
  "description": "The document specifically mentions 'Pok\u00e9mon Gold and Silver (1999, Game Boy Color)', which directly answers the question regarding the release of these games. It provides the necessary information about the title and the year of release, fulfilling the requirements of the query. However, it does not specify the exact date, but since the question does not ask for a specific day or month, the year is sufficient. Therefore, the document is deemed useful for answering the question.",
  "missing_information": ""
}


In [10]:
missing = retrieve_game("Was Mortal Kombat X released for PlayStation 5?")
print("Nearest neighbours for a game that is not in the dataset:")
for doc in missing[:3]:
    print(f"[{doc['source_id']}] sim={doc['similarity']:.3f}  {doc['Name']}")

print("\nEvaluation of a question the database CANNOT answer:")
verdict = evaluate_retrieval(
    question="Was Mortal Kombat X released for PlayStation 5?",
    retrieved_docs=[f"{d['Name']} ({d['YearOfRelease']}, {d['Platform']}) - {d['Description']}" for d in missing],
)
print(json.dumps(verdict, indent=2))

Nearest neighbours for a game that is not in the dataset:
[018] sim=0.472  God of War Ragnarök
[005] sim=0.463  Marvel's Spider-Man 2
[025] sim=0.449  Forza Horizon 5

Evaluation of a question the database CANNOT answer:
{
  "useful": false,
  "confidence": 0.9,
  "description": "The retrieved documents do not contain any information about Mortal Kombat X or its availability on PlayStation 5. They focus on other games, none of which are related to Mortal Kombat X. Since the question specifically asks about a particular game and its platform, and the documents do not address this, they are not useful for answering the query.",
  "missing_information": "Information about Mortal Kombat X and its release on PlayStation 5."
}


In [11]:
# Deliberately NOT the Mortal Kombat X question: this call writes its result to
# long-term memory, which would rob the Query 3 demo below of a cold web fallback.
# A "right now" question also exercises the rule that current events always need the web.
web = game_web_search("What is Rockstar Games working on right now?")

if "error" in web:
    # Surface the reason rather than printing an empty answer.
    print("Web search failed:", web["error"])
else:
    print("answer:", web.get("answer", "")[:400])
    print("\nsources:")
    for result in web.get("results", []):
        print(" -", result["url"])
    print("\nstored in long-term memory as:", web.get("stored_as"))

print("memory fragments:", long_term_memory.count())

answer: Rockstar Games is currently developing Grand Theft Auto 6. No other major projects are publicly announced. The game is expected to release in the future.

sources:
 - https://statusgator.com/services/rockstar-games
 - https://downdetector.com/status/rockstar-games
 - https://www.rockstargames.com/newswire
 - https://www.instagram.com/rockstargames?hl=en

stored in long-term memory as: mem-1e75645107c7
memory fragments: 1


### Agent

The agent is a **state machine** built on `lib/state_machine.py`. Each node does one
thing, and the only branch is the tool loop:

```
        entry
          |
          v
    memory_recall          check long-term memory before spending tokens
          |
          v
  internal_retrieval       ALWAYS: retrieve_game + evaluate_retrieval
          |
          v
    message_prep           system prompt + replayed history + recalled facts + query
          |
          v
    llm_processor  <-----------------------+
          |                                |
   tool calls?                             | tool results appended
     |        \                            |
   yes|         \ no                       |
     v          v                          |
 tool_executor  |  --------------------->--+
                |
                v
       structured_report    validated GameReport JSON with citations
                |
                v
        memory_update       write web findings back to long-term memory
                |
                v
          termination
```

Three things are added on top of the course's `lib/agents.Agent`:

* **`memory_recall`** runs *before* the LLM, so recalled facts are part of the prompt
  rather than something the model has to think to ask for.
* **`internal_retrieval`** runs the first two tools as fixed nodes. Left to
  `tool_choice="auto"`, the model would sometimes answer a games question from its own
  training data and attach a document id it had never been given; as a node, "internal
  knowledge first, then evaluate" is guaranteed on every query and every cited id is one
  the agent actually received.
* **`structured_report`** produces the machine-readable twin of the prose answer. It is
  built from a plain-text transcript of the run rather than from the raw message list, so
  a truncated tool loop can never produce an invalid request.
* **`memory_update`** closes the learning loop.

State is kept at two levels: `ShortTermMemory` holds one `Run` per turn per session
(conversation context, replayed on the next turn), while the ChromaDB memory collection
survives kernel restarts.

An iteration cap stops a model that keeps asking for tools from looping forever.

In [12]:
import json
from typing import Any, Dict, List, Optional, TypedDict, Union




class UdaPlayState(TypedDict):
    """Everything that flows through the state machine for one query."""
    user_query: str
    instructions: str
    session_id: str
    messages: List[Any]
    current_tool_calls: Optional[List[ToolCall]]
    tool_trace: List[Dict[str, Any]]
    memory_context: str
    iteration: int
    total_tokens: int
    retrieval_context: str
    internal_useful: bool
    used_web_search: bool
    final_answer: str
    report: Optional[Dict[str, Any]]


class UdaPlayAgent:
    def __init__(
        self,
        instructions: str,
        tools: List[Tool],
        model_name: str = DEFAULT_LLM_MODEL,
        temperature: float = 0.0,
        max_iterations: int = 6,
        long_term_memory: Optional[LongTermMemoryStore] = None,
        verbose: bool = True,
    ):
        self.instructions = instructions
        self.tools = tools or []
        self.tool_map = {tool.name: tool for tool in self.tools}
        self.model_name = model_name
        self.temperature = temperature
        self.max_iterations = max_iterations
        self.long_term_memory = long_term_memory
        self.verbose = verbose

        self.memory = ShortTermMemory()          # per-session conversation history
        self.workflow = self._create_state_machine()

    # ------------------------------------------------------------------
    # Steps
    # ------------------------------------------------------------------

    def _memory_recall_step(self, state: UdaPlayState) -> UdaPlayState:
        """Look in long-term memory before spending any tokens."""
        if not self.long_term_memory:
            return {"memory_context": "", "iteration": 0}

        try:
            hits = self.long_term_memory.recall(state["user_query"], n_results=3)
        except Exception as exc:  # noqa: BLE001
            if self.verbose:
                print(f"  [memory_recall] skipped: {exc}")
            return {"memory_context": "", "iteration": 0}

        if not hits:
            return {"memory_context": "", "iteration": 0}

        lines = [
            f"- ({hit['id']}, source: {hit['source']}, similarity {hit['similarity']}) {hit['content']}"
            for hit in hits
        ]
        if self.verbose:
            print(f"  [memory_recall] {len(hits)} relevant fragment(s) recalled")
        return {"memory_context": "\n".join(lines), "iteration": 0}

    def _contextualize_query(self, state: UdaPlayState) -> str:
        """
        Rewrite a follow-up into a standalone search query.

        "And which company published it?" is meaningless to a vector search - it
        retrieves five arbitrary games, and whatever the model sees first can hijack
        the referent. Resolving the pronoun against the conversation before searching
        is the standard fix, and it costs one short completion.
        """
        history = [
            message for message in (state.get("messages") or [])
            if getattr(message, "role", "") in ("user", "assistant") and (message.content or "").strip()
        ]
        if not history:
            return state["user_query"]

        transcript = "\n".join(
            f"{message.role}: {truncate(message.content, 300)}" for message in history[-4:]
        )
        prompt = (
            "Rewrite the user's latest message as a standalone search query for a video "
            "game database. Resolve pronouns such as 'it' or 'that game' using the "
            "conversation. If the latest message is already standalone, return it "
            "unchanged. Reply with the query only - no quotes, no explanation.\n\n"
            f"Conversation so far:\n{transcript}\n\n"
            f"Latest message: {state['user_query']}\n\nStandalone query:"
        )
        try:
            rewritten = (SafeLLM(model=self.model_name, temperature=0.0)
                         .invoke(prompt).content or "").strip()
        except Exception:  # noqa: BLE001 - fall back to the raw query
            return state["user_query"]
        return rewritten or state["user_query"]

    def _internal_retrieval_step(self, state: UdaPlayState) -> UdaPlayState:
        """
        Always search the internal database first, then judge the result.

        The rubric's workflow - internal knowledge, then evaluation, then web
        fallback - is a guarantee, not a suggestion the model is free to skip. With
        `tool_choice="auto"` the model sometimes answered games questions straight
        from its own training data and attached a document id it had never been
        given. Running `retrieve_game` and `evaluate_retrieval` as fixed nodes makes
        both steps happen on every single query, and means a cited id is always an id
        the agent actually received.
        """
        retrieve = self.tool_map.get("retrieve_game")
        evaluate = self.tool_map.get("evaluate_retrieval")
        if retrieve is None:
            return {"retrieval_context": "", "internal_useful": False}

        query = self._contextualize_query(state)
        trace = list(state.get("tool_trace") or [])
        if self.verbose and query != state["user_query"]:
            print(f"  [internal_retrieval] follow-up resolved to: {query}")

        try:
            hits = retrieve(query=query)
        except Exception as exc:  # noqa: BLE001
            if self.verbose:
                print(f"  [internal_retrieval] retrieval failed: {exc}")
            return {"retrieval_context": "", "internal_useful": False}

        trace.append({
            "iteration": 0,
            "tool": "retrieve_game",
            "arguments": {"query": query},
            "result": truncate(self._serialize(hits), 600),
        })

        documents = [
            f"[{hit['source_id']}] {hit['Name']} ({hit['YearOfRelease']}, {hit['Platform']}) - "
            f"genre {hit['Genre']}, publisher {hit['Publisher']}. {hit['Description']}"
            for hit in hits
        ]

        verdict: Dict[str, Any] = {}
        if evaluate is not None:
            try:
                verdict = evaluate(question=query, retrieved_docs=documents)
            except Exception as exc:  # noqa: BLE001
                verdict = {
                    "useful": False,
                    "confidence": 0.0,
                    "description": f"The evaluator failed ({type(exc).__name__}). Treating the retrieval as insufficient.",
                    "missing_information": "",
                }
            trace.append({
                "iteration": 0,
                "tool": "evaluate_retrieval",
                "arguments": {"question": query, "retrieved_docs": documents},
                "result": truncate(self._serialize(verdict), 600),
            })

        useful = bool(verdict.get("useful", False))
        listed = "\n".join(
            f"- {document} (similarity {hit['similarity']})"
            for document, hit in zip(documents, hits)
        ) or "- (nothing found)"

        context = (
            f"Internal game database - already searched for you with the query "
            f"\"{query}\". These are the results:\n"
            f"{listed}\n\n"
            f"Retrieval evaluation: useful={useful}, confidence={verdict.get('confidence', 'n/a')}\n"
            f"{verdict.get('description', '')}"
        )
        if verdict.get("missing_information"):
            context += f"\nMissing from the internal documents: {verdict['missing_information']}"

        if self.verbose:
            print(f"  [internal_retrieval] {len(hits)} document(s) retrieved, useful={useful}")

        return {"retrieval_context": context, "internal_useful": useful, "tool_trace": trace}

    def _prepare_messages_step(self, state: UdaPlayState) -> UdaPlayState:
        """Seed the system prompt, replay history, inject memory, add the query."""
        messages = list(state.get("messages") or [])

        if not messages:
            messages = [SystemMessage(content=state["instructions"])]

        if state.get("memory_context"):
            messages.append(SystemMessage(content=(
                "Facts you previously learned and stored in long-term memory. "
                "They may answer the question without a new web search, but verify "
                "they are relevant before relying on them:\n"
                f"{state['memory_context']}"
            )))

        if state.get("retrieval_context"):
            messages.append(SystemMessage(content=state["retrieval_context"]))

        messages.append(UserMessage(content=state["user_query"]))
        return {"messages": messages}

    def _llm_step(self, state: UdaPlayState) -> UdaPlayState:
        """One LLM turn: either a final answer or a batch of tool calls."""
        llm = SafeLLM(
            model=self.model_name,
            temperature=self.temperature,
            tools=self.tools,
        )
        response = llm.invoke(state["messages"])
        tool_calls = response.tool_calls or None

        total_tokens = state.get("total_tokens", 0)
        if response.token_usage:
            total_tokens += response.token_usage.total_tokens

        ai_message = AIMessage(content=response.content, tool_calls=tool_calls)

        if self.verbose:
            if tool_calls:
                names = ", ".join(call.function.name for call in tool_calls)
                print(f"  [llm] requested tool(s): {names}")
            else:
                print("  [llm] produced a final answer")

        return {
            "messages": state["messages"] + [ai_message],
            "current_tool_calls": tool_calls,
            "iteration": state.get("iteration", 0) + 1,
            "total_tokens": total_tokens,
            "final_answer": response.content or state.get("final_answer", ""),
        }

    def _tool_step(self, state: UdaPlayState) -> UdaPlayState:
        """Execute every pending tool call and feed the results back."""
        tool_messages: List[ToolMessage] = []
        trace = list(state.get("tool_trace") or [])
        used_web_search = state.get("used_web_search", False)

        for call in state.get("current_tool_calls") or []:
            name = call.function.name
            try:
                arguments = json.loads(call.function.arguments or "{}")
            except json.JSONDecodeError:
                arguments = {}

            tool = self.tool_map.get(name)
            if tool is None:
                result: Any = {"error": f"Unknown tool `{name}`"}
            else:
                try:
                    result = tool(**arguments)
                except Exception as exc:  # noqa: BLE001 - surface errors to the LLM
                    result = {"error": f"{type(exc).__name__}: {exc}"}

            if name == "game_web_search":
                failed = isinstance(result, dict) and "error" in result
                used_web_search = used_web_search or not failed

            content = self._serialize(result)
            tool_messages.append(ToolMessage(
                content=content, tool_call_id=call.id, name=name,
            ))
            trace.append({
                "iteration": state.get("iteration", 0),
                "tool": name,
                "arguments": arguments,
                "result": truncate(content, 600),
            })
            if self.verbose:
                print(f"  [tool] {name}({json.dumps(arguments)[:120]}) -> {truncate(content, 160)}")

        return {
            "messages": state["messages"] + tool_messages,
            "current_tool_calls": None,
            "tool_trace": trace,
            "used_web_search": used_web_search,
        }

    def _structured_report_step(self, state: UdaPlayState) -> UdaPlayState:
        """Turn the finished conversation into a validated GameReport."""
        messages = self._sanitize(state["messages"])
        transcript = self._build_transcript(state)

        prompt = (
            "You are formatting the result of a completed research task into JSON.\n"
            "Use ONLY the information in the transcript below - do not add facts.\n"
            "Cite every source that contributed: internal_db for document ids like `006`, "
            "web for URLs, long_term_memory for ids starting with `mem-`.\n"
            "Include a source ONLY if it actually supports the answer. The retrieval "
            "results in the transcript are frequently unrelated to it - on a follow-up "
            "question the answer often comes from earlier in the conversation instead. "
            "When that happens, return an empty sources list rather than citing documents "
            "that do not support the answer, and say so in the answer text.\n"
            "List in `games` only the games the answer is actually about, not every game "
            "that happened to appear in the retrieval results.\n"
            "For each game, `platform` must be a platform the game was actually released "
            "on, not one it merely runs on through backwards compatibility.\n"
            "Set confidence to `high` when a source directly states the answer, "
            "`medium` when it is inferred, and `low` when the answer is uncertain.\n\n"
            f"{transcript}"
        )

        report: Optional[Dict[str, Any]] = None
        try:
            llm = SafeLLM(model=self.model_name, temperature=0.0)
            response = llm.invoke(prompt, response_format=GameReport)
            report = PydanticOutputParser(model_class=GameReport).parse(response).model_dump()
        except Exception as exc:  # noqa: BLE001 - never lose the prose answer
            if self.verbose:
                print(f"  [report] structured output unavailable ({type(exc).__name__}: {exc})")
            report = GameReport(
                question=state["user_query"],
                answer=state.get("final_answer", ""),
                confidence="low",
                used_web_search=state.get("used_web_search", False),
            ).model_dump()

        return {"messages": messages, "report": report}

    def _memory_update_step(self, state: UdaPlayState) -> UdaPlayState:
        """Write anything learned from the web back to long-term memory."""
        if not self.long_term_memory or not state.get("used_web_search"):
            return {}

        answer = state.get("final_answer") or ""
        if not answer.strip():
            return {}

        sources = [
            citation.get("reference", "")
            for citation in (state.get("report") or {}).get("sources", [])
            if citation.get("source_type") == "web"
        ]
        try:
            fragment_id = self.long_term_memory.remember(
                content=f"Q: {state['user_query']}\nA: {answer}",
                source="; ".join(sources) or "web_search",
                question=state["user_query"],
            )
            if self.verbose:
                print(f"  [memory_update] stored web finding as {fragment_id}")
        except Exception as exc:  # noqa: BLE001
            if self.verbose:
                print(f"  [memory_update] skipped: {exc}")
        return {}

    # ------------------------------------------------------------------
    # Wiring
    # ------------------------------------------------------------------

    def _create_state_machine(self) -> StateMachine[UdaPlayState]:
        machine = StateMachine[UdaPlayState](UdaPlayState)

        entry = EntryPoint[UdaPlayState]()
        memory_recall = Step[UdaPlayState]("memory_recall", self._memory_recall_step)
        internal_retrieval = Step[UdaPlayState]("internal_retrieval", self._internal_retrieval_step)
        message_prep = Step[UdaPlayState]("message_prep", self._prepare_messages_step)
        llm_processor = Step[UdaPlayState]("llm_processor", self._llm_step)
        tool_executor = Step[UdaPlayState]("tool_executor", self._tool_step)
        structured_report = Step[UdaPlayState]("structured_report", self._structured_report_step)
        memory_update = Step[UdaPlayState]("memory_update", self._memory_update_step)
        termination = Termination[UdaPlayState]()

        machine.add_steps([
            entry, memory_recall, internal_retrieval, message_prep, llm_processor,
            tool_executor, structured_report, memory_update, termination,
        ])

        machine.connect(entry, memory_recall)
        machine.connect(memory_recall, internal_retrieval)
        machine.connect(internal_retrieval, message_prep)
        machine.connect(message_prep, llm_processor)

        def route_after_llm(state: UdaPlayState) -> Union[Step, str]:
            """Keep looping while the model wants tools, with an iteration cap."""
            if state.get("current_tool_calls") and state.get("iteration", 0) < self.max_iterations:
                return tool_executor
            return structured_report

        machine.connect(llm_processor, [tool_executor, structured_report], route_after_llm)
        machine.connect(tool_executor, llm_processor)
        machine.connect(structured_report, memory_update)
        machine.connect(memory_update, termination)

        return machine

    # ------------------------------------------------------------------
    # Public API
    # ------------------------------------------------------------------

    def invoke(self, query: str, session_id: Optional[str] = None) -> Run:
        """Answer one query, carrying over the conversation for that session."""
        session_id = session_id or "default"
        self.memory.create_session(session_id)

        previous_messages: List[Any] = []
        last_run: Optional[Run] = self.memory.get_last_object(session_id)
        if last_run and last_run.get_final_state():
            previous_messages = self._sanitize(last_run.get_final_state().get("messages", []))

        initial_state: UdaPlayState = {
            "user_query": query,
            "instructions": self.instructions,
            "session_id": session_id,
            "messages": previous_messages,
            "current_tool_calls": None,
            "tool_trace": [],
            "memory_context": "",
            "retrieval_context": "",
            "internal_useful": False,
            "iteration": 0,
            "total_tokens": 0,
            "used_web_search": False,
            "final_answer": "",
            "report": None,
        }

        run = self.workflow.run(initial_state)
        self.memory.add(run, session_id)
        return run

    def get_session_runs(self, session_id: Optional[str] = None) -> List[Run]:
        return self.memory.get_all_objects(session_id or "default")

    def reset_session(self, session_id: Optional[str] = None):
        self.memory.reset(session_id)

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _serialize(result: Any) -> str:
        if isinstance(result, BaseModel):
            return result.model_dump_json()
        if isinstance(result, (dict, list)):
            return json.dumps(result, default=str, ensure_ascii=False)
        return str(result)

    @staticmethod
    def _sanitize(messages: List[Any]) -> List[Any]:
        """
        Drop a trailing assistant message that still has unanswered tool calls.

        That only happens when the iteration cap fires mid-loop, but replaying
        such a history on the next turn would be rejected by the API, which
        requires every tool call to be followed by its tool result.
        """
        if messages and getattr(messages[-1], "tool_calls", None):
            last = messages[-1]
            messages = messages[:-1] + [AIMessage(
                content=last.content or "(stopped before these tool calls could run)"
            )]
        return messages

    def _build_transcript(self, state: UdaPlayState) -> str:
        parts = [f"USER QUESTION: {state['user_query']}"]
        if state.get("memory_context"):
            parts.append(f"LONG-TERM MEMORY:\n{state['memory_context']}")
        for entry in state.get("tool_trace") or []:
            parts.append(
                f"TOOL `{entry['tool']}` called with {json.dumps(entry['arguments'], default=str)}\n"
                f"RESULT: {entry['result']}"
            )
        parts.append(f"FINAL ANSWER GIVEN TO THE USER:\n{state.get('final_answer', '')}")
        return "\n\n".join(parts)

#### Instructions

The system prompt encodes the retrieval policy the rubric asks for - internal knowledge
first, evaluate, then fall back - and the citation format. Everything the agent must do
on *every* turn lives here rather than in the per-question prompts.

In [13]:
INSTRUCTIONS = """You are UdaPlay, an AI research assistant for the video game industry.
You answer questions about games, release dates, platforms, genres and publishers.

# Workflow
1. The internal game database has ALREADY been searched for this question before your
   turn, and the results plus a retrieval evaluation are in the context above. That is
   step one of the workflow, and it is done for you on every question.
2. If the evaluation says the documents are useful, answer from them.
3. If it says they are NOT useful:
   a. Call `recall_learned_facts` - you may already have learned the answer before.
   b. If memory does not settle it, call `game_web_search`.
   c. Answer from the web results, and say plainly if they are still inconclusive.
4. Call `retrieve_game` yourself only when the pre-fetched search clearly used the wrong
   wording - for example a follow-up question that did not name the game. Follow it with
   `evaluate_retrieval` on the new documents.
5. Questions about current, ongoing, upcoming or unreleased things always need
   `game_web_search`. The internal database is a static snapshot and cannot know them.

# Citations - required
- Internal documents: cite the source_id in square brackets, e.g. [006].
- Web results: cite the url, e.g. [https://example.com/page].
- Long-term memory: cite the fragment id, e.g. [mem-1a2b3c4d].
- Cite ONLY ids and urls that actually appear in the retrieval results, memory fragments
  or tool output you were given. Never guess a document id, and never answer a games
  question from your own background knowledge without a source to point at.

# Style
- Answer in 1-4 sentences. Lead with the fact the user asked for.
- Include platform, year and publisher when they are relevant to the question.
- Distinguish a native release on a platform from backwards compatibility: "playable on
  PS5 through backwards compatibility" is not the same claim as "released for PS5".
- State your confidence when sources disagree or are thin.
- Never invent a game, date, platform or publisher. If neither the internal database nor
  the web answers the question, say so directly.
- Use the conversation history: follow-up questions like "and on which platform?" refer
  to the game discussed in the previous turn. The internal search results are retrieved
  from the user's message and may be about the wrong games on a follow-up - trust the
  conversation over them, and call `retrieve_game` yourself with the resolved game name
  if you need documents for it.
- If nothing in the message or the conversation identifies which game is being asked
  about, ask the user which game they mean. Never pick one of the retrieved games at
  random and answer about that instead.
"""

agent = UdaPlayAgent(
    instructions=INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search, recall_learned_facts],
    model_name=DEFAULT_LLM_MODEL,
    temperature=0.0,
    max_iterations=6,
    long_term_memory=long_term_memory,
)

print(agent.workflow)
print("Tools:", [t.name for t in agent.tools])

StateMachine(schema=['user_query', 'instructions', 'session_id', 'messages', 'current_tool_calls', 'tool_trace', 'memory_context', 'iteration', 'total_tokens', 'retrieval_context', 'internal_useful', 'used_web_search', 'final_answer', 'report'])
Tools: ['retrieve_game', 'evaluate_retrieval', 'game_web_search', 'recall_learned_facts']


### Reporting

`print_report` renders one run: the path the state machine took, every tool call with its
arguments and result, the prose answer, and the structured JSON with citations.

In [14]:
def print_report(run: Run, show_state_path: bool = True) -> Dict[str, Any]:
    """Render a completed run: reasoning path, tool usage, answer and citations."""
    state = run.get_final_state()
    report = state.get("report") or {}

    print("=" * 100)
    print(f"QUESTION: {state['user_query']}")
    print("=" * 100)

    if show_state_path:
        path = " -> ".join(snapshot.step_id.strip("_") for snapshot in run.snapshots)
        print(f"\nSTATE MACHINE PATH\n  {path}")

    if state.get("memory_context"):
        print("\nLONG-TERM MEMORY RECALLED")
        for line in state["memory_context"].splitlines():
            print(f"  {line}")

    if state.get("retrieval_context"):
        print(f"\nINTERNAL RETRIEVAL\n  documents judged useful: {state.get('internal_useful')}")

    print(f"\nTOOL USAGE ({len(state.get('tool_trace') or [])} call(s))")
    for index, entry in enumerate(state.get("tool_trace") or [], start=1):
        print(f"  {index}. {entry['tool']}({json.dumps(entry['arguments'], ensure_ascii=False)[:180]})")
        print(f"     -> {truncate(entry['result'], 320)}")

    print("\nFINAL ANSWER")
    print(f"  {state.get('final_answer', '')}")

    if report:
        print(f"\nCONFIDENCE: {report.get('confidence')}   "
              f"WEB SEARCH USED: {report.get('used_web_search')}")
        print("\nCITATIONS")
        if report.get("sources"):
            for source in report["sources"]:
                print(f"  - [{source['source_type']}] {source['reference']}"
                      + (f" - {source['detail']}" if source.get("detail") else ""))
        else:
            print("  (none reported)")

        print("\nSTRUCTURED OUTPUT (JSON)")
        print(json.dumps(report, indent=2, ensure_ascii=False))

    print(f"\nTokens used: {state.get('total_tokens', 0)}   "
          f"LLM turns: {state.get('iteration', 0)}")
    print()
    return report

### Example queries

Three questions that exercise the paths through the agent:

1. a release-date question the internal database covers at year granularity,
2. a fact it holds but only implicitly, requiring reasoning over the retrieved documents,
3. a fact it does not hold at all, forcing the web fallback.

All three run in the session `demo`, so each turn can see the ones before it.

Query 1 is worth watching either way it lands. The internal document gives the release
*year*; whether that counts as answering "when was it released" is a genuine judgement
call, and the evaluator does not always make it the same way. When it accepts the year,
the answer comes back from `[006]` with no web call. When it wants an exact date, the
fallback fires and the answer comes back from the web with a fuller date. Both outcomes
are the two-tier system behaving correctly - the escalation is the point.

#### Query 1 - a release date the internal database covers by year

In [15]:
run_1 = agent.invoke("When was Pokémon Gold and Silver released?", session_id="demo")
report_1 = print_report(run_1)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
  [internal_retrieval] 5 document(s) retrieved, useful=False
[StateMachine] Executing step: internal_retrieval
[StateMachine] Executing step: message_prep
  [llm] requested tool(s): game_web_search
[StateMachine] Executing step: llm_processor
  [tool] game_web_search({"question": "What is the exact release date for Pok\u00e9mon Gold and Silver?"}) -> {"question": "What is the exact release date for Pokémon Gold and Silver?", "answer": "Pokémon Gold and Silver were released in Japan on November 21, 1999, a...
[StateMachine] Executing step: tool_executor
  [llm] produced a final answer
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: structured_report
  [memory_update] stored web finding as mem-292c3b8b0ecd
[StateMachine] Executing step: memory_update
[StateMachine] Terminating: __termination__
QUESTION: When was Pokémon Gold and Silver released?

STATE MACHINE PATH
  entry -> memor

#### Query 2 - requires reasoning over the retrieved documents

In [16]:
run_2 = agent.invoke("Which one was the first 3D platformer Mario game?", session_id="demo")
report_2 = print_report(run_2)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
  [internal_retrieval] 5 document(s) retrieved, useful=True
[StateMachine] Executing step: internal_retrieval
[StateMachine] Executing step: message_prep
  [llm] produced a final answer
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: structured_report
[StateMachine] Executing step: memory_update
[StateMachine] Terminating: __termination__
QUESTION: Which one was the first 3D platformer Mario game?

STATE MACHINE PATH
  entry -> memory_recall -> internal_retrieval -> message_prep -> llm_processor -> structured_report -> memory_update

INTERNAL RETRIEVAL
  documents judged useful: True

TOOL USAGE (2 call(s))
  1. retrieve_game({"query": "Which one was the first 3D platformer Mario game?"})
     -> [{"source_id": "009", "Name": "Super Mario 64", "Platform": "Nintendo 64", "YearOfRelease": 1996, "Genre": "Platformer", "Publisher": "Nintendo", "Description": "A groundbreaking 3D plat

#### Query 3 - not in the database, so the agent must fall back to the web

In [17]:
run_3 = agent.invoke("Was Mortal Kombat X released for Playstation 5?", session_id="demo")
report_3 = print_report(run_3)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
  [internal_retrieval] follow-up resolved to: Was Mortal Kombat X released for PlayStation 5?
  [internal_retrieval] 5 document(s) retrieved, useful=False
[StateMachine] Executing step: internal_retrieval
[StateMachine] Executing step: message_prep
  [llm] requested tool(s): game_web_search
[StateMachine] Executing step: llm_processor
  [tool] game_web_search({"question": "Was Mortal Kombat X released for PlayStation 5?"}) -> {"question": "Was Mortal Kombat X released for PlayStation 5?", "answer": "Mortal Kombat X was not originally released for PlayStation 5. It is playable on P...
[StateMachine] Executing step: tool_executor
  [llm] produced a final answer
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: structured_report
  [memory_update] stored web finding as mem-900369c4fbde
[StateMachine] Executing step: memory_update
[StateMachine] Terminating: __termination__
QUESTION: Wa

### Conversation state

The agent is stateful: each turn replays the previous turns of its session. The follow-up
below names no game at all, so it can only be answered from context.

In [18]:
run_4 = agent.invoke("And which company published it?", session_id="demo")
report_4 = print_report(run_4)

print(f"Turns stored in session `demo`: {len(agent.get_session_runs('demo'))}")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
  [internal_retrieval] follow-up resolved to: Which company published Mortal Kombat X?
  [internal_retrieval] 5 document(s) retrieved, useful=False
[StateMachine] Executing step: internal_retrieval
[StateMachine] Executing step: message_prep
  [llm] requested tool(s): game_web_search
[StateMachine] Executing step: llm_processor
  [tool] game_web_search({"question": "Which company published Mortal Kombat X?"}) -> {"question": "Which company published Mortal Kombat X?", "answer": "Mortal Kombat X was published by Warner Bros. Interactive Entertainment. The game was rel...
[StateMachine] Executing step: tool_executor
  [llm] produced a final answer
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: structured_report
  [memory_update] stored web finding as mem-1338da0ba8c1
[StateMachine] Executing step: memory_update
[StateMachine] Terminating: __termination__
QUESTION: And which compan

In [19]:
# A different session starts clean - proof that sessions are isolated.
run_isolated = agent.invoke("And which company published it?", session_id="fresh")
print("\nAnswer without any conversation history:")
print(" ", run_isolated.get_final_state()["final_answer"])

[StateMachine] Starting: __entry__
  [memory_recall] 2 relevant fragment(s) recalled
[StateMachine] Executing step: memory_recall
  [internal_retrieval] 5 document(s) retrieved, useful=False
[StateMachine] Executing step: internal_retrieval
[StateMachine] Executing step: message_prep
  [llm] produced a final answer
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: structured_report
[StateMachine] Executing step: memory_update
[StateMachine] Terminating: __termination__

Answer without any conversation history:
  Could you please specify which game you are asking about?


### Long-term memory

Everything the agent learned from the web during this notebook is in the persistent
memory collection. Because it lives in ChromaDB rather than in the kernel, it is still
there after a restart - so asking the Mortal Kombat X question again starts from a
`memory_recall` hit instead of a cold web search.

In [20]:
fragments = long_term_memory.all_fragments()
print(f"{len(fragments)} fragment(s) in long-term memory\n")
for fragment in fragments:
    print(f"[{fragment['id']}] source: {fragment.get('source', '')}")
    print(f"  {truncate(fragment['content'], 300)}\n")

7 fragment(s) in long-term memory

[mem-1e75645107c7] source: https://statusgator.com/services/rockstar-games
  What is Rockstar Games working on right now? -> Rockstar Games is currently developing Grand Theft Auto 6. No other major projects are publicly announced. The game is expected to release in the future.

[mem-c859ce6c0b96] source: https://en.wikipedia.org/wiki/Pok%C3%A9mon_Gold_and_Silver
  What is the exact release date for Pokémon Gold and Silver? -> Pokémon Gold and Silver were released in Japan on November 21, 1999, and in North America on October 15, 2000.

[mem-292c3b8b0ecd] source: https://en.wikipedia.org/wiki/Pok%C3%A9mon_Gold_and_Silver
  Q: When was Pokémon Gold and Silver released?
A: Pokémon Gold and Silver were released in Japan on November 21, 1999, and in North America on October 15, 2000 [https://en.wikipedia.org/wiki/Pok%C3%A9mon_Gold_and_Silver].

[mem-225946a0adc5] source: https://www.youtube.com/watch?v=-JPXripEMoA
  Was Mortal Kombat X released for PlaySt

In [21]:
run_5 = agent.invoke(
    "Remind me - could you play Mortal Kombat X on a PlayStation 5?",
    session_id="memory-demo",
)
report_5 = print_report(run_5)

[StateMachine] Starting: __entry__
  [memory_recall] 3 relevant fragment(s) recalled
[StateMachine] Executing step: memory_recall
  [internal_retrieval] 5 document(s) retrieved, useful=False
[StateMachine] Executing step: internal_retrieval
[StateMachine] Executing step: message_prep
  [llm] produced a final answer
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: structured_report
[StateMachine] Executing step: memory_update
[StateMachine] Terminating: __termination__
QUESTION: Remind me - could you play Mortal Kombat X on a PlayStation 5?

STATE MACHINE PATH
  entry -> memory_recall -> internal_retrieval -> message_prep -> llm_processor -> structured_report -> memory_update

LONG-TERM MEMORY RECALLED
  - (mem-900369c4fbde, source: https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved, similarity 0.6515) Q: Was Mortal Kombat X released for Playstation 5?
  A: Mortal Kombat X was not originally released for PlayStation 5, but it is playable on PS5 

### Automated evaluation

`lib/evaluation.py` provides an LLM judge for the final answer. Pairing it with a check
of the actual tool trace covers both halves of agent quality: did it get the right
answer, and did it get there the right way (internal first, web only when needed).

In [22]:
from lib.evaluation import AgentEvaluator, TestCase

test_cases = [
    TestCase(
        id="tc-01",
        description="Release year of a game held in the internal database",
        user_query="When was Pokémon Gold and Silver released?",
        expected_tools=["retrieve_game", "evaluate_retrieval"],
        reference_answer="Pokémon Gold and Silver released in 1999 on the Game Boy Color.",
    ),
    TestCase(
        id="tc-02",
        description="Identify the first 3D Mario platformer from internal documents",
        user_query="Which one was the first 3D platformer Mario game?",
        expected_tools=["retrieve_game", "evaluate_retrieval"],
        reference_answer="Super Mario 64, released in 1996 for the Nintendo 64.",
    ),
    TestCase(
        id="tc-03",
        description="Game absent from the database - must fall back to the web",
        user_query="Was Mortal Kombat X realeased for Playstation 5?",
        expected_tools=["retrieve_game", "evaluate_retrieval", "game_web_search"],
        reference_answer="No. Mortal Kombat X came out in 2015 for PS4, Xbox One and PC; "
                         "there was no native PlayStation 5 release.",
    ),
]

evaluator = AgentEvaluator()
rows = []

for test_case in test_cases:
    started = time.time()
    run = agent.invoke(test_case.user_query, session_id=f"eval-{test_case.id}")
    elapsed = time.time() - started

    state = run.get_final_state()
    tools_used = [entry["tool"] for entry in state.get("tool_trace") or []]
    expected_covered = all(name in tools_used for name in test_case.expected_tools)
    # A web-fallback case is also satisfied when the answer came out of long-term
    # memory, which is the cheaper path the agent is supposed to prefer.
    from_memory = bool(state.get("memory_context")) and not expected_covered
    route_ok = expected_covered or from_memory

    result = evaluator.evaluate_final_response(
        test_case=test_case,
        agent_response=state.get("final_answer", ""),
        execution_time=elapsed,
        total_tokens=state.get("total_tokens", 0),
    )

    rows.append({
        "id": test_case.id,
        "score": result.overall_score,
        "completed": result.task_completion.task_completed,
        "expected_tools_used": route_ok,
        "route": "memory" if (from_memory and not expected_covered) else "tools",
        "tools": tools_used,
        "tokens": state.get("total_tokens", 0),
        "seconds": round(elapsed, 1),
        "feedback": result.feedback,
    })

print(f"{'id':<7} {'score':<7} {'done':<6} {'route ok':<9} {'via':<8} {'tokens':<8} {'secs':<6} tools used")
print("-" * 110)
for row in rows:
    print(f"{row['id']:<7} {row['score']:<7.2f} {str(row['completed']):<6} "
          f"{str(row['expected_tools_used']):<9} {row['route']:<8} {row['tokens']:<8} "
          f"{row['seconds']:<6} {', '.join(row['tools'])}")

print(f"\nMean score: {sum(r['score'] for r in rows) / len(rows):.2f}")
print("\nJudge feedback")
for row in rows:
    print(f"  {row['id']}: {truncate(row['feedback'], 300)}")

[StateMachine] Starting: __entry__
  [memory_recall] 2 relevant fragment(s) recalled
[StateMachine] Executing step: memory_recall
  [internal_retrieval] 5 document(s) retrieved, useful=False
[StateMachine] Executing step: internal_retrieval
[StateMachine] Executing step: message_prep
  [llm] produced a final answer
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: structured_report
[StateMachine] Executing step: memory_update
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
  [internal_retrieval] 5 document(s) retrieved, useful=True
[StateMachine] Executing step: internal_retrieval
[StateMachine] Executing step: message_prep
  [llm] produced a final answer
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: structured_report
[StateMachine] Executing step: memory_update
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
  [memory_recall

### Summary

**Tools**

| Tool | Role | Backed by |
|---|---|---|
| `retrieve_game` | primary retrieval | ChromaDB `udaplay` collection |
| `evaluate_retrieval` | quality gate, returns `useful` + `confidence` | LLM-as-judge, validated into `EvaluationReport` |
| `game_web_search` | fallback | Tavily, writes findings to long-term memory |
| `recall_learned_facts` | cheaper alternative to re-searching | ChromaDB memory collection |

**Agent**

* State machine with seven nodes and a capped tool loop.
* Two layers of state: per-session conversation history (`ShortTermMemory`) and
  cross-session knowledge (persistent ChromaDB memory).
* Dual output: prose plus a schema-validated `GameReport` carrying typed citations,
  a confidence level and a `used_web_search` flag.

**Behaviour observed above**

* Internal-knowledge questions are answered without touching the web.
* A question about a game outside the dataset is caught by the evaluation step and
  routed to the web, with URLs cited.
* Context carries across turns inside a session, and sessions are isolated from each
  other.
* Web findings persist, so the same question asked later is served from memory.

**Possible extensions**

* Cache the retrieval judge's verdicts to cut token use on repeated queries.
* Add a freshness timestamp check so stale memory fragments are re-verified.
* Expand the dataset with review scores and add a sentiment tool over them.